# Bring your model: a MONAI UNet in KonfAI, ten lines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fideus-labs/KonfAI/blob/main/examples/BringYourModel/BringYourModel_demo.ipynb)

A model you already have (here MONAI's `UNet`, untouched) trains and predicts on a KonfAI dataset through two calls, `konfai.train_model` and `konfai.predict_model`, with the patching, the overlap blending, the streamed writes and the run record every KonfAI run keeps. No YAML: the config the run would have read is built from the arguments and kept in the workspace as the run's record.

The data is the Segmentation example's: five pelvis CT cases with a 41-label reference (~114 MB, from `VBoussot/konfai-demo`). The training is deliberately short; the scores demonstrate the pipeline, not the method.


In [ ]:
# Setup: find KonfAI (cloning it on Colab), install what is missing, load the notebook helpers.
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/fideus-labs/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples").is_dir())
sys.path.insert(0, str(REPO_DIR / "examples"))

from konfai_demo import read, setup, show

EXAMPLE_DIR, DATASET_DIR, DEVICE = setup(REPO_DIR, "BringYourModel", ("konfai", f"{REPO_DIR}[imaging]"), "huggingface_hub", "matplotlib", "monai")
GPU = [int(DEVICE[DEVICE.index("--gpu") + 1])] if "--gpu" in DEVICE else None


## 1. The data

The same five cases as the Segmentation example, one folder per case: `Dataset/<case>/CT.mha` and `Dataset/<case>/SEG.mha`. The dataset root is spelled the way a YAML would spell it, `./Dataset:mha`, and the groups are the file names: `CT` is what the model reads, `SEG` what it is scored on.


In [ ]:
import shutil

from huggingface_hub import snapshot_download

if not any(DATASET_DIR.glob("*/CT.mha")):
    snapshot_download("VBoussot/konfai-demo", repo_type="dataset", allow_patterns="Segmentation/**", local_dir=str(DATASET_DIR))
    for case in (DATASET_DIR / "Segmentation").iterdir():
        shutil.move(str(case), DATASET_DIR / case.name)
    shutil.rmtree(DATASET_DIR / "Segmentation")
    shutil.rmtree(DATASET_DIR / ".cache", ignore_errors=True)

CASES = sorted(path.name for path in DATASET_DIR.iterdir() if path.is_dir())
print(len(CASES), "cases:", ", ".join(CASES))


## 2. Your model

Any `torch.nn.Module` with one tensor in and one out. This one is MONAI's 2D `UNet`, built as MONAI builds it; KonfAI wraps it for execution and exposes its output under the name `Model`. It is fed slices of `[1, 256, 256]` (the patch's non-unit axes say the model is 2D), one channel in, 41 classes out.


In [ ]:
import os

import torch
from monai.networks.nets import UNet

os.chdir(EXAMPLE_DIR)  # the dataset root and the workspaces below are relative to the example directory
model = UNet(spatial_dims=2, in_channels=1, out_channels=41, channels=(32, 64, 128, 256), strides=(2, 2, 2), num_res_units=2)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")


## 3. Train

The ten lines. `loss` is a KonfAI criterion attached to the model's output; `transforms` are the same stage objects a YAML would name (the reference labels are cast to integers for the loss; the CT is standardized per case); `validation` holds one case in five out. The run writes `Checkpoints/MONAI_UNET/` and `Statistics/MONAI_UNET/` (the TensorBoard curves and `Trainer.yml`, the resolved config, which names the model by a token: a live model runs in the process that built it).


In [ ]:
import konfai
from konfai.data.transform import Standardize, TensorCast
from konfai.metric.measure import CrossEntropyLoss, Dice

checkpoints = konfai.train_model(
    model,
    "./Dataset:mha",
    inputs="CT",
    targets="SEG",
    loss=[CrossEntropyLoss(), Dice(labels=list(range(1, 41)))],
    patch=[1, 256, 256],
    epochs=3,
    batch_size=8,
    lr=1e-3,
    transforms={"CT": [Standardize()], "SEG": [TensorCast(dtype="int64")]},
    validation=0.2,
    autocast=GPU is not None,
    name="MONAI_UNET",
    manual_seed=32,
    gpu=GPU,
    overwrite=True,
)
BEST = sorted(checkpoints.glob("*.pt"))[-1]
print("checkpoint:", BEST)


## 4. Predict

The checkpoint the training wrote, over the same cases, slice by slice with the input's own geometry. The 41 logit channels are reduced to a label map by `Argmax` before the write, and stored as bytes. The output lands under the run's workspace, `Predictions/MONAI_UNET/Pred/<case>/PRED.mha`.


In [ ]:
from konfai.data.transform import Argmax

workspace = konfai.predict_model(
    model,
    "./Dataset:mha",
    inputs="CT",
    patch=[1, 256, 256],
    output="./Pred:mha",
    checkpoints=BEST,
    transforms={"CT": [Standardize()]},
    final_transforms=[Argmax(), TensorCast(dtype="uint8")],
    autocast=GPU is not None,
    name="MONAI_UNET",
    gpu=GPU,
    overwrite=True,
)
print("predictions under", workspace / "Pred")


## 5. Look at one case

The CT, the reference labels and the prediction on the most annotated slice. Three epochs on four cases: the shapes are there, the boundaries are not; raise `epochs` for a real result.


In [ ]:
import numpy as np

case = CASES[0]
ct, seg = read(DATASET_DIR / case / "CT.mha"), read(DATASET_DIR / case / "SEG.mha")
pred = read(workspace / "Pred" / case / "PRED.mha")
LABEL_MAX = int(seg.max())
best = int(np.argmax((seg > 0).sum(axis=(1, 2))))
show([(f"CT, slice {best}", ct[best], "gray"), ("reference SEG", ct[best], "gray", seg[best]), ("MONAI UNet, 3 epochs", ct[best], "gray", pred[best])], LABEL_MAX)


## What just happened, and where to go next

- `konfai.train_model` and `konfai.predict_model` built the config tree KonfAI reads (`Statistics/MONAI_UNET/Trainer.yml` shows it) and ran the same engine as the YAML route: patch sampling, overlap-blended reassembly, streamed writes, the checkpoint format, the run record.
- Left without `checkpoints`, `predict_model` writes the weights the module holds in memory as a checkpoint, so a model loaded any other way (a library's pretrained weights, a foreign checkpoint) predicts as it stands.
- A live model runs on one rank, in this process. For several GPUs, or to resume a training, spell the model as a classpath (`monai.networks.nets:UNet`) in a `Config.yml`: the {doc}`Segmentation example <../Segmentation/README>` is that route, and `Statistics/MONAI_UNET/Trainer.yml` is a starting point for it.
